# 04 · Clasificación supervisada del rezago escolar | ENAHO + ESCALE 2025

**Objetivo:** comparar una **regresión logística** (baseline) con **Random Forest**, **XGBoost** y **LightGBM**, sin contaminación entre datos de entrenamiento y prueba.

**Requisitos de la guía que atendemos:** separación final **GroupShuffleSplit** por `CONGLOME`; selección y ajuste con **GroupKFold**; imputación, escalado y one-hot *dentro* del pipeline; ponderación de clases y `FACTOR07` **solamente en cada conjunto de entrenamiento**; evaluación con ROC-AUC, PR-AUC (AP), F1 macro, matriz de confusión y calibración. Se fija `random_state=42`.

**Interpretación:** `REZAGO_ESCOLAR` es la **etiqueta operativa** documentada en fase 02, *no* la estimación oficial del MINEDU/INEI. El resultado predictivo tampoco representa una relación causal. Usamos exclusivamente los seis archivos originales ya integrados.

**Importante:** en esta fase **NO se modifica ni se duplica el dataset maestro**. Se guardan reportes, figuras, identificadores de filas de prueba y el pipeline ganador para la siguiente fase SHAP.

> Ejecutar desde la carpeta `notebooks/` o la raíz del proyecto con **Kernel → Restart & Run All**. Si una librería falta, ejecutar primero la instalación de la siguiente celda.

## Paso 0 — Instalar dependencias (solo una vez)

Ejecuta la siguiente línea en una celda de Jupyter **si** aparece `ModuleNotFoundError` en la importación; después reinicia el kernel:

```python
%pip install -U pandas numpy scikit-learn matplotlib xgboost lightgbm joblib
```

No usamos `pyarrow` ni necesitaremos instalar un lector DBF: leeremos el CSV maestro que ya construiste.

## Paso 1 — Importación, rutas y lectura del dataset maestro

In [1]:
from pathlib import Path
import json, warnings, platform
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sklearn, xgboost, lightgbm, joblib
from IPython.display import display
from sklearn.model_selection import GroupShuffleSplit, GroupKFold
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from sklearn.metrics import (roc_auc_score, average_precision_score, f1_score,
                             precision_score, recall_score, confusion_matrix,
                             roc_curve, precision_recall_curve, brier_score_loss,
                             ConfusionMatrixDisplay)
from sklearn.calibration import calibration_curve

SEMILLA = 42
cwd = Path.cwd().resolve()
candidatos = []
for base in (cwd, *cwd.parents):
    candidatos.extend((base, base/'proyecto_brecha_digital_2025'))
ROOT = next((b for b in candidatos if (b/'data'/'processed'/'dataset_enaho_escale_2025.csv').exists()), None)
if ROOT is None:
    raise FileNotFoundError('No se encontró data/processed/dataset_enaho_escale_2025.csv. Abre este notebook desde tu carpeta del proyecto.')
RUTA = ROOT/'data'/'processed'/'dataset_enaho_escale_2025.csv'
REPORTES = ROOT/'reportes'
FIG = REPORTES/'figuras'
MODELOS = ROOT/'modelos'
for d in (REPORTES, FIG, MODELOS):
    d.mkdir(parents=True, exist_ok=True)

dtypes = {c:str for c in ['AÑO','MES','CONGLOME','VIVIENDA','HOGAR','CODPERSO','UBIGEO']}
df = pd.read_csv(RUTA, encoding='utf-8-sig', low_memory=False, dtype=dtypes)
print('✓ Proyecto:', ROOT)
print('✓ Maestro:', df.shape)
print('✓ Clustering disponible:', 'CLUSTER_PERFIL' in df.columns)
print('✓ Objetivo disponible:', 'REZAGO_ESCOLAR' in df.columns)
assert len(df) == 23645, 'La cantidad de filas no coincide con el control del proyecto.'
assert 'CONGLOME' in df and 'REZAGO_ESCOLAR' in df
assert set(pd.to_numeric(df.REZAGO_ESCOLAR,errors='coerce').dropna().unique()).issubset({0,1})


ModuleNotFoundError: No module named 'xgboost'

## Paso 2 — Universo elegible y matriz de predictores sin fuga

**Excluimos** `EDAD`, `P208A`, `GRADO_EBR`, `EDAD_NORMATIVA`, `NIVEL_EBR`, `P308A`, `P308B`, `P308C`, `REZAGO_ESCOLAR`, todas las llaves de persona/vivienda/hogar y `CLUSTER_PERFIL`. La etiqueta de rezago se calculó a partir de edad y grado: incluir estos campos daría una predicción trivial por fuga. El clustering fue ajustado con todos los registros; por eso también queda fuera.

Ingresos y gastos se transforman con `log1p` determinístico (sin estimar parámetros). Los códigos de TIC, sexo, lengua materna y pobreza se tratan como **categorías**, no magnitudes; sus significados definitivos siguen sujetos al diccionario ENAHO. La imputación y codificación **se ajustarán dentro de cada fold**, no aquí.

In [ ]:
y_all = pd.to_numeric(df['REZAGO_ESCOLAR'], errors='coerce')
elegibles = y_all.isin([0,1])
sub = df.loc[elegibles].copy()
idx_original = df.index[elegibles].to_numpy()
y = y_all.loc[elegibles].astype(int).reset_index(drop=True)
grupos = sub['CONGLOME'].astype(str).reset_index(drop=True)
assert not grupos.isna().any() and (~grupos.isin(['', 'nan', 'None'])).all()

NUMERICAS_ORIG = ['INGRESO_POR_MIEMBRO','GASTO_POR_MIEMBRO','MIEPERHO',
                   'ESCALE_DIST_RATIO_ALUMNOS_DOCENTE','ESCALE_DIST_PROP_PUBLICOS',
                   'ESCALE_DIST_N_COLEGIOS']
CATEGORICAS = ['P207','P300A','POBREZA','AREA_OPERATIVA_ENAHO',
               'INTERNET_HOGAR','ELECTRICIDAD_HOGAR','USO_INTERNET_MES']
faltan = [c for c in NUMERICAS_ORIG+CATEGORICAS+['FACTOR07'] if c not in sub]
if faltan: raise KeyError('Faltan variables en el maestro: '+str(faltan))
X = pd.DataFrame(index=range(len(sub)))
for c in NUMERICAS_ORIG:
    v = pd.to_numeric(sub[c],errors='coerce').reset_index(drop=True)
    if c in ['INGRESO_POR_MIEMBRO','GASTO_POR_MIEMBRO','ESCALE_DIST_N_COLEGIOS']:
        v = np.log1p(v.where(v>=0))
        X['LOG_'+c] = v
    else:
        X[c] = v.replace([np.inf,-np.inf],np.nan)
NUMERICAS = [c for c in X.columns]
for c in CATEGORICAS:
    # str categórico; conservar NaN como NaN, y no convertir códigos a magnitudes
    v = sub[c].reset_index(drop=True)
    X[c] = v.map(lambda z: str(z) if pd.notna(z) else np.nan).astype(object)
expansion = pd.to_numeric(sub['FACTOR07'],errors='coerce').reset_index(drop=True)
expansion = expansion.where(expansion>0, 1.0).fillna(1.0).astype(float)
assert len(X)==len(y)==len(grupos)==len(idx_original)
assert 'CLUSTER_PERFIL' not in X and 'EDAD' not in X and 'REZAGO_ESCOLAR' not in X
assert y.value_counts().min()>100 and grupos.nunique()>100
print('✓ N para aprendizaje supervisado:',len(y))
print('✓ Grupos de conglomerado:',grupos.nunique())
print('✓ Etiquetas:',y.value_counts().sort_index().to_dict())
print('✓ Predictores numéricos:',NUMERICAS)
print('✓ Predictores categóricos:',CATEGORICAS)
print('✓ Exclusiones críticas contra fuga: comprobadas')


## Paso 3 — Separación definitiva por conglomerado (20 % de grupos para prueba)

Separamos **una vez**, antes de cualquier imputación, escalado, codificación o ajuste de modelo. El conjunto de prueba **no interviene** en la selección de hiperparámetros ni del ganador.

In [ ]:
holdout=GroupShuffleSplit(n_splits=1,test_size=0.20,random_state=SEMILLA)
itr,ite=next(holdout.split(X,y,groups=grupos))
Xtr,Xte = X.iloc[itr],X.iloc[ite]
ytr,yte = y.iloc[itr],y.iloc[ite]
gtr,gte = grupos.iloc[itr],grupos.iloc[ite]
wtr,wte = expansion.iloc[itr],expansion.iloc[ite]
assert set(gtr).isdisjoint(set(gte)), '¡Fuga: conglomerados compartidos entre train y test!'
assert set(ytr.unique())=={0,1} and set(yte.unique())=={0,1}
print(f'Entrenamiento: {len(itr):,} | Test aislado: {len(ite):,}')
print(f'Conglomerados train: {gtr.nunique()} | Test: {gte.nunique()} | Intersección: {len(set(gtr)&set(gte))}')
print('Distribución y train:',ytr.value_counts(normalize=True).sort_index().round(3).to_dict())
print('Distribución y test: ',yte.value_counts(normalize=True).sort_index().round(3).to_dict())


## Paso 4 — Preprocesamiento interno + pesos de cada fold

`ColumnTransformer` combina imputación de mediana y estandarización para numéricas con imputación de moda y `OneHotEncoder` para categóricas. El `fit` sucede solo en los datos de entrenamiento de cada fold.

**Pesos de entrenamiento:** combinamos factor muestral `FACTOR07` y pesos inversos a las frecuencias de clase calculadas **solamente desde `y` del fold de entrenamiento**. Los pesos se normalizan a media 1. El test se evalúa sin ponderación (las métricas predictivas no son estimaciones poblacionales).

In [ ]:
def construir_preprocesador():
    try:
        ohe=OneHotEncoder(handle_unknown='ignore',sparse_output=False)
    except TypeError:  # compatibilidad con sklearn anterior
        ohe=OneHotEncoder(handle_unknown='ignore',sparse=False)
    return ColumnTransformer([
        ('num',Pipeline([('imputar',SimpleImputer(strategy='median')),
                         ('escalar',StandardScaler())]),NUMERICAS),
        ('cat',Pipeline([('imputar',SimpleImputer(strategy='most_frequent')),
                         ('codificar',ohe)]),CATEGORICAS)
    ],remainder='drop',sparse_threshold=0)

def pesos_entrenamiento(y_fold, factor_fold):
    yf=np.asarray(y_fold,dtype=int)
    wf=np.asarray(factor_fold,dtype=float)
    frec=np.bincount(yf,minlength=2).astype(float)
    if (frec==0).any(): raise ValueError('Fold con una sola clase; no se puede ponderar.')
    p_clase=len(yf)/(2*frec)
    peso=wf*p_clase[yf]
    return peso/np.mean(peso)

def metricas(y_real, probabilidades):
    pred=(np.asarray(probabilidades)>=0.5).astype(int)
    return {'ROC_AUC':float(roc_auc_score(y_real,probabilidades)),
            'PR_AUC_AP':float(average_precision_score(y_real,probabilidades)),
            'F1_MACRO':float(f1_score(y_real,pred,average='macro',zero_division=0)),
            'PRECISION_REZAGO':float(precision_score(y_real,pred,zero_division=0)),
            'RECALL_REZAGO':float(recall_score(y_real,pred,zero_division=0)),
            'BRIER':float(brier_score_loss(y_real,probabilidades))}

print('✓ Transformaciones y pesos preparados sin ajuste global.')


## Paso 5 — GroupKFold y comparación de hiperparámetros

Cada candidato se evalúa con exactamente los **mismos tres folds** de entrenamiento. Se comparan una configuración baseline y **dos configuraciones por modelo avanzado**; el mejor ajuste por algoritmo se selecciona por PR-AUC/AP en CV. `max_depth`, número de árboles y regularización son parámetros predefinidos. No se usa el test para optimizarlos.

> Este paso puede llevar varios minutos dependiendo de tu PC; no cierres Jupyter durante la ejecución.

In [ ]:
CANDIDATOS=[
 ('Logistica','baseline',dict(C=1.0)),
 ('RandomForest','rf_180',dict(n_estimators=180,max_depth=10,min_samples_leaf=5)),
 ('RandomForest','rf_260',dict(n_estimators=260,max_depth=14,min_samples_leaf=4)),
 ('XGBoost','xgb_160',dict(n_estimators=160,max_depth=3,learning_rate=0.065)),
 ('XGBoost','xgb_240',dict(n_estimators=240,max_depth=4,learning_rate=0.055)),
 ('LightGBM','lgb_160',dict(n_estimators=160,num_leaves=15,learning_rate=0.06)),
 ('LightGBM','lgb_240',dict(n_estimators=240,num_leaves=31,learning_rate=0.045))]

def construir_modelo(nombre,parametros):
    if nombre=='Logistica':
        return LogisticRegression(max_iter=1200,random_state=SEMILLA,**parametros)
    if nombre=='RandomForest':
        return RandomForestClassifier(n_jobs=-1,random_state=SEMILLA,**parametros)
    if nombre=='XGBoost':
        return XGBClassifier(objective='binary:logistic',eval_metric='logloss',
                             tree_method='hist',n_jobs=2,random_state=SEMILLA,**parametros)
    if nombre=='LightGBM':
        return LGBMClassifier(objective='binary',verbosity=-1,n_jobs=2,
                              random_state=SEMILLA,**parametros)
    raise KeyError(nombre)

gkf=GroupKFold(n_splits=3)
folds=list(gkf.split(Xtr,ytr,groups=gtr))
for ia,ib in folds:
    assert set(gtr.iloc[ia]).isdisjoint(set(gtr.iloc[ib]))
registros=[]
for nombre,config,params in CANDIDATOS:
    for nfold,(ia,iv) in enumerate(folds,1):
        pipeline=Pipeline([('pre',construir_preprocesador()),('modelo',construir_modelo(nombre,params))])
        pesos=pesos_entrenamiento(ytr.iloc[ia],wtr.iloc[ia])
        pipeline.fit(Xtr.iloc[ia],ytr.iloc[ia],modelo__sample_weight=pesos)
        probs=pipeline.predict_proba(Xtr.iloc[iv])[:,1]
        m=metricas(ytr.iloc[iv],probs)
        registros.append({'MODELO':nombre,'CONFIG':config,'FOLD':nfold,**m})
    print(f'✓ {nombre:13s} {config:10s} — PR-AUC: '+str(round(np.mean([r['PR_AUC_AP'] for r in registros if r['CONFIG']==config]),4)))
cv=pd.DataFrame(registros)
cv.to_csv(REPORTES/'04_cv_folds_2025.csv',index=False,encoding='utf-8-sig')
res=cv.groupby(['MODELO','CONFIG'],as_index=False).agg(
    ROC_AUC=('ROC_AUC','mean'),PR_AUC_AP=('PR_AUC_AP','mean'),
    F1_MACRO=('F1_MACRO','mean'),BRIER=('BRIER','mean'),
    ROC_STD=('ROC_AUC','std'),AP_STD=('PR_AUC_AP','std'))
res=res.sort_values(['PR_AUC_AP','F1_MACRO'],ascending=False).reset_index(drop=True)
mejores=res.sort_values(['PR_AUC_AP','F1_MACRO'],ascending=False).drop_duplicates('MODELO')
res.to_csv(REPORTES/'04_cv_configuraciones_2025.csv',index=False,encoding='utf-8-sig')
mejores.to_csv(REPORTES/'04_cv_comparacion_modelos_2025.csv',index=False,encoding='utf-8-sig')
print('\nComparación CV por modelo (solo train):')
display(mejores.round(4))


## Paso 6 — Selección del ganador SOLO por validación cruzada

Se decide sin consultar las etiquetas del conjunto de prueba. Para la comparación final, se reentrena cada uno de los cuatro modelos usando todo train. El ganador se guarda para SHAP.

In [ ]:
ganador=mejores.iloc[0]
MODELO_GANADOR=str(ganador.MODELO)
CONFIG_GANADORA=str(ganador.CONFIG)
params_seleccionados={}
for row in mejores.itertuples(index=False):
    params_seleccionados[row.MODELO]=next(p for n,c,p in CANDIDATOS if n==row.MODELO and c==row.CONFIG)
print('✓ Ganador elegido con CV:',MODELO_GANADOR,CONFIG_GANADORA,
      '| PR-AUC/AP=',round(float(ganador.PR_AUC_AP),4))


## Paso 7 — Evaluación final en test aislado (una vez)

Se reportan todas las métricas y la confusión para cada arquitectura. El resultado de prueba no cambia la elección realizada mediante CV.

In [ ]:
pruebas=[]
probas_test={}
modelo_final_ganador=None
pesos_finales=pesos_entrenamiento(ytr,wtr)
for nombre,params in params_seleccionados.items():
    pipe=Pipeline([('pre',construir_preprocesador()),('modelo',construir_modelo(nombre,params))])
    pipe.fit(Xtr,ytr,modelo__sample_weight=pesos_finales)
    p=pipe.predict_proba(Xte)[:,1]
    probas_test[nombre]=p
    pruebas.append({'MODELO':nombre,**metricas(yte,p)})
    if nombre==MODELO_GANADOR:
        modelo_final_ganador=pipe
        joblib.dump(pipe,MODELOS/'04_modelo_ganador_rezago_2025.joblib')
test_tab=pd.DataFrame(pruebas).sort_values('PR_AUC_AP',ascending=False)
test_tab.to_csv(REPORTES/'04_test_modelos_2025.csv',index=False,encoding='utf-8-sig')
print('✓ Modelos evaluados en test aislado:')
display(test_tab.round(4))
print('✓ Modelo seleccionado ANTES de mirar test:',MODELO_GANADOR)
print('✓ Modelo guardado:',MODELOS/'04_modelo_ganador_rezago_2025.joblib')


## Paso 8 — Gráficos ROC y precisión-recall (comparación final)

In [ ]:
fig,axs=plt.subplots(1,2,figsize=(12,4.8))
for nombre,probs in probas_test.items():
    fpr,tpr,_=roc_curve(yte,probs)
    prec,rec,_=precision_recall_curve(yte,probs)
    met=next(r for r in pruebas if r['MODELO']==nombre)
    axs[0].plot(fpr,tpr,label=f'{nombre} (AUC={met["ROC_AUC"]:.3f})')
    axs[1].plot(rec,prec,label=f'{nombre} (AP={met["PR_AUC_AP"]:.3f})')
axs[0].plot([0,1],[0,1],'--',alpha=.6)
axs[1].axhline(float(yte.mean()),linestyle='--',alpha=.6,label='Prevalencia test')
axs[0].set(title='Curvas ROC — test aislado',xlabel='Tasa falsos positivos',ylabel='Sensibilidad')
axs[1].set(title='Curvas precisión-recall — test aislado',xlabel='Recall',ylabel='Precisión')
for ax in axs: ax.legend(fontsize=8);ax.grid(alpha=.15)
fig.tight_layout()
fig.savefig(FIG/'04_roc_pr_modelos.png',dpi=180,bbox_inches='tight')
plt.show()


## Paso 9 — Matriz de confusión y análisis de falsos positivos/negativos

In [ ]:
p_win=probas_test[MODELO_GANADOR]
pred_win=(p_win>=0.5).astype(int)
cm=confusion_matrix(yte,pred_win,labels=[0,1])
fig,ax=plt.subplots(figsize=(6,5))
ConfusionMatrixDisplay(cm,display_labels=['Sin rezago','Con rezago']).plot(ax=ax,values_format='d',cmap='Blues',colorbar=False)
ax.set_title('Matriz de confusión · '+MODELO_GANADOR+' (test)')
fig.tight_layout()
fig.savefig(FIG/'04_matriz_confusion.png',dpi=180,bbox_inches='tight')
plt.show()
tn,fp,fn,tp=cm.ravel()
print('Verdaderos negativos:',tn,'Falsos positivos:',fp,
      'Falsos negativos:',fn,'Verdaderos positivos:',tp)
print('Un falso negativo representa un estudiante con rezago al que el modelo asigna bajo riesgo.')


## Paso 10 — Calibración de probabilidades (test aislado)

In [ ]:
fig,ax=plt.subplots(figsize=(7,5))
ax.plot([0,1],[0,1],'--',color='gray',label='Calibración perfecta')
for nombre,p in probas_test.items():
    obs,est=calibration_curve(yte,p,n_bins=10,strategy='quantile')
    ax.plot(est,obs,marker='o',label=nombre)
ax.set(title='Curva de calibración — test aislado',xlabel='Probabilidad estimada',
       ylabel='Frecuencia observada',xlim=(0,1),ylim=(0,1))
ax.legend(fontsize=8)
ax.grid(alpha=.2)
fig.tight_layout()
fig.savefig(FIG/'04_calibracion.png',dpi=180,bbox_inches='tight')
plt.show()
print('Nota: una probabilidad discriminativa no implica necesariamente una probabilidad calibrada.')


## Paso 11 — Guardar controles y preparación para SHAP

Se exportan **solo índices originales** de filas de test, sin copiar el dataset maestro. SHAP leerá el **mismo** archivo procesado y cargará el pipeline ganador; los identificadores de test nunca se usan durante el ajuste.

In [ ]:
test_indices=pd.DataFrame({'IDX_MAESTRO':idx_original[ite].astype(int)})
test_indices.to_csv(REPORTES/'04_indices_test_2025.csv',index=False)
control={
 'fase':'04_clasificacion_supervisada','anio':2025,'random_state':SEMILLA,
 'total_maestro':int(len(df)),'universo_etiquetado':int(len(y)),
 'sin_etiqueta':int((~elegibles).sum()),
 'n_entrenamiento':int(len(itr)),'n_test':int(len(ite)),
 'grupos_entrenamiento':int(gtr.nunique()),'grupos_test':int(gte.nunique()),
 'interseccion_conglomerados':int(len(set(gtr)&set(gte))),
 'cv':'GroupKFold(3)','test':'GroupShuffleSplit(test_size=0.20)',
 'num_modelos':len(params_seleccionados),'mejor_modelo_por_cv':MODELO_GANADOR,
 'config_ganadora':CONFIG_GANADORA,
 'cv_ganador':{k:float(ganador[k]) for k in ['ROC_AUC','PR_AUC_AP','F1_MACRO','BRIER']},
 'test_ganador':next(r for r in pruebas if r['MODELO']==MODELO_GANADOR),
 'confusion':{'TN':int(tn),'FP':int(fp),'FN':int(fn),'TP':int(tp)},
 'predictores_numericos':NUMERICAS,'predictores_categoricos':CATEGORICAS,
 'prohibidos':['CLUSTER_PERFIL','REZAGO_ESCOLAR','EDAD','P208A','GRADO_EBR','EDAD_NORMATIVA','NIVEL_EBR',
              'P308A','P308B','P308C','CONGLOME','UBIGEO','CODPERSO'],
 'ponderacion_entrenamiento':'FACTOR07 * frecuencia inversa de clase calculada dentro del fold (normalizada)',
 'ponderacion_evaluacion':'métricas no ponderadas; no son estimaciones nacionales',
 'nota':'Variable objetivo operativa fase 02, no tasa oficial. Códigos TIC deben validarse con cuestionario/diccionario.'
}
assert len(test_indices)==len(ite)
assert len(set(gtr)&set(gte))==0
assert all(c not in X for c in control['prohibidos'])
with open(REPORTES/'04_control_supervisado_2025.json','w',encoding='utf-8') as f:
    json.dump(control,f,ensure_ascii=False,indent=2)
print('✓ Reportes en:',REPORTES)
print('✓ Gráficas en:',FIG)
print('✓ Test aislado y anti-fuga validados')
print('✓ Próximo paso: 05_SHAP_explicabilidad_2025.ipynb')
print('✓ Maestro intacto:',RUTA)


## Comprobación de entregables

- `reportes/04_cv_folds_2025.csv` — 3 folds por configuración.
- `reportes/04_cv_configuraciones_2025.csv` — búsqueda de hiperparámetros.
- `reportes/04_cv_comparacion_modelos_2025.csv` — un mejor ajuste por arquitectura.
- `reportes/04_test_modelos_2025.csv` — evaluación final.
- `reportes/04_control_supervisado_2025.json` — auditoría anti-fuga.
- `reportes/04_indices_test_2025.csv` — filas para análisis SHAP posterior.
- `reportes/figuras/04_roc_pr_modelos.png`, `04_matriz_confusion.png`, `04_calibracion.png`.
- `modelos/04_modelo_ganador_rezago_2025.joblib` — pipeline completo entrenado.

Para el informe científico, recuerda: las cifras de clustering y las métricas de clasificación se refieren a la muestra ENAHO/ESCALE analizada, no constituyen causalidad ni estimación nacional automáticamente. **No vuelvas a usar `CLUSTER_PERFIL` como predictor.**